<a href="https://colab.research.google.com/github/Alejandro120325/Practica_02_IA/blob/main/Practica_02_Preprocesamiento_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 02: Preprocesamiento de Datos

**Nombre:** Jairo Alejandro Ojeda Herrera  
**Asignatura:** Inteligencia Artificial  
**Nivel:** 6to Nivel  
**Fecha:** 18/10/2026  

En esta práctica se trabaja con clasificación de variables, escalado de datos y preprocesamiento usando NumPy, Pandas y scikit-learn.


## Verificación de librerías


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

print("Todas las librerías cargadas correctamente")


# FASE 1. Anatomía de un Dataset


## Dataset de estudiantes


In [ ]:
datos = {
    "nombre": ["Ana", "Luis", "María", "Carlos", "Sofía", "Diego"],
    "edad": [21, 23, 20, 25, 22, 21],
    "promedio_gpa": [8.9, 7.4, 9.5, 6.1, 8.2, 7.8],
    "horas_estudio": [18, 10, 22, 6, 15, 12],
    "distancia_km": [2.5, 15.0, 0.8, 40.2, 5.1, 8.7],
    "carrera": ["Computación", "Computación", "Redes", "Redes", "Computación", "Redes"],
    "nivel_ingles": ["Básico", "Intermedio", "Avanzado", "Básico", "Avanzado", "Intermedio"],
    "beca": [1, 0, 1, 0, 1, 0],
    "aprobado": ["Sí", "No", "Sí", "No", "Sí", "No"]
}

df = pd.DataFrame(datos)

print(df)
print(f"\nDimensiones del dataset: {df.shape}")


## Actividad 1.1. Identificación de la estructura

**P1.** El dataset tiene 6 instancias. Cada fila representa a un estudiante.

**P2.** Los atributos son: `nombre`, `edad`, `promedio_gpa`, `horas_estudio`, `distancia_km`, `carrera`, `nivel_ingles`, `beca` y `aprobado`.

Si el objetivo es predecir si un estudiante aprueba:

- **Features:** edad, promedio_gpa, horas_estudio, distancia_km, carrera, nivel_ingles y beca.
- **Label:** aprobado.

**P3.** `nombre` no debería usarse como feature porque identifica a la persona, pero no representa una característica útil para generalizar la predicción.


## Actividad 1.2. Clasificación de variables


In [ ]:
clasificacion = {
    "columna": [
        "edad", "promedio_gpa", "horas_estudio", "distancia_km",
        "carrera", "nivel_ingles", "beca", "aprobado"
    ],
    "cuantitativa_o_cualitativa": [
        "Cuantitativa", "Cuantitativa", "Cuantitativa", "Cuantitativa",
        "Cualitativa", "Cualitativa", "Cualitativa", "Cualitativa"
    ],
    "discreta_o_continua": [
        "Discreta", "Continua", "Discreta", "Continua",
        "NA", "NA", "NA", "NA"
    ],
    "nominal_u_ordinal": [
        "NA", "NA", "NA", "NA",
        "Nominal", "Ordinal", "Nominal", "Nominal"
    ]
}

df_tipos = pd.DataFrame(clasificacion)
print(df_tipos)


**P4.** `nivel_ingles` es una variable cualitativa ordinal porque sus categorías tienen un orden: Básico, Intermedio y Avanzado. Esta diferencia importa porque puede codificarse respetando ese orden.

**P5.** `beca` es cualitativa nominal binaria. Aunque utiliza 0 y 1, esos números representan categorías: no tiene beca y sí tiene beca.


## Actividad 1.3. Inspección de tipos de datos


In [ ]:
print("Tipos de datos detectados por pandas:")
print(df.dtypes)

print("\nEstadísticas descriptivas:")
print(df.describe())


**P6.** Los tipos de Pandas no siempre coinciden con la clasificación conceptual. Por ejemplo, `beca` aparece como `int64`, pero conceptualmente es una variable cualitativa binaria.

**P7.** `distancia_km` presenta la mayor desviación estándar. Si se usa un algoritmo basado en distancias, como KNN, una variable con una escala mucho mayor puede dominar el cálculo.


# FASE 2. Escalado con NumPy


## Actividad 2.1. Preparación de los datos


In [ ]:
columnas_numericas = ["promedio_gpa", "horas_estudio", "distancia_km"]
X = df[columnas_numericas].values.astype(float)

print("Matriz de datos original:")
print(X)
print(f"\nForma: {X.shape} -> {X.shape[0]} instancias, {X.shape[1]} features")


**P8.** Se excluye `beca` porque ya está representada con 0 y 1 y normalmente no necesita escalado. `edad` es numérica, pero en esta fase se trabaja únicamente con las tres variables seleccionadas para comparar escalas continuas.


## Actividad 2.2. Normalización Min-Max


In [ ]:
def min_max_manual(X):
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    X_norm = (X - X_min) / (X_max - X_min)
    return X_norm, X_min, X_max

X_minmax, mins, maxs = min_max_manual(X)

print("RESULTADO MIN-MAX")
print(f"{'Feature':<18} {'Min original':>14} {'Max original':>14}")
print("-" * 48)

for i, col in enumerate(columnas_numericas):
    print(f"{col:<18} {mins[i]:>14.2f} {maxs[i]:>14.2f}")

df_minmax = pd.DataFrame(
    X_minmax,
    columns=[c + "_mm" for c in columnas_numericas]
)

print("\nDatos normalizados:")
print(df_minmax.round(4))


**P9.** Después de Min-Max, cada columna queda entre 0 y 1. Un valor de 0 representa el mínimo original de esa columna y un valor de 1 representa el máximo.

**P10.** Si aparece un nuevo valor extremo, por ejemplo 80 km, el máximo cambia y los valores anteriores se comprimen hacia 0. Esto muestra que Min-Max es sensible a valores extremos.


## Actividad 2.3. Estandarización Z-Score


In [ ]:
def zscore_manual(X):
    media = X.mean(axis=0)
    std = X.std(axis=0)
    X_z = (X - media) / std
    return X_z, media, std

X_zscore, medias, stds = zscore_manual(X)

print("RESULTADO Z-SCORE")
print(f"{'Feature':<18} {'Media':>10} {'Desv.Est.':>12}")
print("-" * 42)

for i, col in enumerate(columnas_numericas):
    print(f"{col:<18} {medias[i]:>10.3f} {stds[i]:>12.3f}")

df_z = pd.DataFrame(
    X_zscore,
    columns=[c + "_z" for c in columnas_numericas]
)

print("\nDatos estandarizados:")
print(df_z.round(4))


**P11.** Con Z-Score, los datos se centran alrededor de una media cercana a 0 y una desviación estándar cercana a 1.

**P12.** El valor de Carlos en `horas_estudio` es negativo porque está por debajo de la media del grupo. No significa que estudie "mal"; solo indica su posición relativa frente a los demás.

**P13.** Para `distancia_km`, Min-Max muestra que Carlos está en el máximo del grupo, mientras que Z-Score también indica cuántas desviaciones estándar se aleja de la media. Por eso Z-Score resulta más informativo para analizar qué tan extremo es ese valor.


## Actividad 2.4. Comparación visual


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
nombres = df["nombre"].values

for i, col in enumerate(columnas_numericas):
    axes[i, 0].bar(nombres, X[:, i])
    axes[i, 0].set_title(f"{col} - Original")
    axes[i, 0].set_ylabel("Valor")

    axes[i, 1].bar(nombres, X_minmax[:, i])
    axes[i, 1].set_title(f"{col} - Min-Max [0,1]")

    axes[i, 2].bar(nombres, X_zscore[:, i])
    axes[i, 2].axhline(0, linestyle="--", linewidth=1)
    axes[i, 2].set_title(f"{col} - Z-Score")

plt.suptitle("Comparación: Original vs Min-Max vs Z-Score")
plt.tight_layout()
plt.show()


**P14.** Las proporciones y el orden relativo de los estudiantes se mantienen. Lo que cambia es la escala utilizada para representar los datos.


## Actividad 2.5. Simulación de un outlier


In [ ]:
X_con_outlier = np.vstack([X, [8.5, 14, 120.0]])

X_mm_con_outlier, _, _ = min_max_manual(X_con_outlier)
X_z_con_outlier, _, _ = zscore_manual(X_con_outlier)

print("Columna distancia_km con outlier de 120 km")
print(f"{'Estudiante':<10} {'Original':>10} {'Min-Max':>10} {'Z-Score':>10}")
print("-" * 44)

nombres_ext = list(nombres) + ["Nuevo"]

for j in range(len(nombres_ext)):
    print(
        f"{nombres_ext[j]:<10} "
        f"{X_con_outlier[j, 2]:>10.1f} "
        f"{X_mm_con_outlier[j, 2]:>10.4f} "
        f"{X_z_con_outlier[j, 2]:>10.4f}"
    )


**P15.** Con Min-Max, el outlier hace que los demás valores se compriman mucho. Z-Score también se ve afectado porque usa la media y la desviación estándar, aunque en este ejemplo el efecto es menor. Para datos con outliers, `RobustScaler` suele ser una mejor opción porque usa mediana e IQR.


# FASE 3. Dataset real con Pandas: Palmer Penguins


## Carga del dataset


In [ ]:
url = "https://huggingface.co/datasets/SIH/palmer-penguins/resolve/main/penguins.csv"
df_penguins = pd.read_csv(url)

print("PRIMERA VISTA DEL DATASET")
print(df_penguins.head(8))

print(f"\nDimensiones: {df_penguins.shape}")
print(f"\nColumnas: {list(df_penguins.columns)}")


## Actividad 3.1. Exploración inicial


In [ ]:
print("TIPOS DE DATOS")
print(df_penguins.dtypes)

print("\nVALORES NULOS")
print(df_penguins.isnull().sum())

print("\nESTADÍSTICAS DESCRIPTIVAS")
print(df_penguins.describe())

print("\nVALORES ÚNICOS EN COLUMNAS CATEGÓRICAS")
for col in df_penguins.select_dtypes(include="object").columns:
    print(f"{col}: {df_penguins[col].unique()}")


**P16.** El dataset tiene 344 instancias. Las variables numéricas incluyen `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` y `year`. Para una tarea de clasificación, el label más lógico es `species`.

**P17.** Existen valores nulos en algunas mediciones y en `sex`. Es importante tratarlos antes del escalado porque pueden propagarse en los cálculos y causar problemas en etapas posteriores.


## Actividad 3.2. Clasificación de variables

| Columna | Tipo |
|---|---|
| species | Cualitativa nominal |
| island | Cualitativa nominal |
| bill_length_mm | Cuantitativa continua |
| bill_depth_mm | Cuantitativa continua |
| flipper_length_mm | Cuantitativa continua |
| body_mass_g | Cuantitativa continua |
| sex | Cualitativa nominal |
| year | Cuantitativa discreta |

**P18.** `sex` e `island` son nominales porque sus categorías no tienen un orden natural. Si se crea otra variable ordenando las islas por temperatura, esa nueva variable sí tendría un orden, pero `island` por sí sola sigue siendo nominal.


## Actividad 3.3. Preprocesamiento


In [ ]:
df_clean = df_penguins.dropna()

print(
    f"Filas originales: {len(df_penguins)} "
    f"-> Filas limpias: {len(df_clean)}"
)

features_num = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

X_penguins = df_clean[features_num].values

print("\nEstadísticas antes del escalado:")
print(
    pd.DataFrame(
        X_penguins,
        columns=features_num
    ).describe().round(2)
)


**P19.** `body_mass_g` tiene una escala mucho mayor que `bill_depth_mm`. En algoritmos basados en distancias, esto puede hacer que el peso corporal domine el cálculo aunque no necesariamente sea la variable más importante.


## Actividad 3.4. Escalado y distribuciones


In [ ]:
mm = MinMaxScaler()
X_mm = mm.fit_transform(X_penguins)

zs = StandardScaler()
X_zs = zs.fit_transform(X_penguins)

print("Min-Max")
print(pd.DataFrame(X_mm, columns=features_num).describe().round(4))

print("\nZ-Score")
print(pd.DataFrame(X_zs, columns=features_num).describe().round(4))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(X_penguins[:, 3], bins=20)
axes[0].set_title("body_mass_g - Original")

axes[1].hist(X_mm[:, 3], bins=20)
axes[1].set_title("body_mass_g - Min-Max")

axes[2].hist(X_zs[:, 3], bins=20)
axes[2].set_title("body_mass_g - Z-Score")

plt.tight_layout()
plt.show()


**P20.** El escalado no cambia la forma de la distribución; cambia la escala y la ubicación de los valores en el eje horizontal.

**P21.** `body_mass_g` no forma una única campana perfecta porque existen grupos asociados a especie y sexo. Z-Score sigue siendo útil para estandarizar, mientras que Min-Max es útil cuando se necesita trabajar en un rango fijo.


# FASE 4. Hugging Face y scikit-learn


## Actividad 4.1. Dataset limpio


In [ ]:
X = df_clean[features_num].values

print(f"Dataset listo: {X.shape[0]} instancias, {X.shape[1]} features")
print("\nEstadísticas originales:")
print(pd.DataFrame(X, columns=features_num).describe().round(2))


In [ ]:
rangos = X.max(axis=0) - X.min(axis=0)

df_rangos = pd.DataFrame({
    "feature": features_num,
    "rango": rangos
})

print(df_rangos)

indice_mayor = np.argmax(rangos)
indice_menor = np.argmin(rangos)

print("\nMayor rango:", features_num[indice_mayor], rangos[indice_mayor])
print("Menor rango:", features_num[indice_menor], rangos[indice_menor])


**P22.** La feature con mayor rango es `body_mass_g` y la de menor rango es `bill_depth_mm`. El código anterior calcula los valores exactos.


## Actividad 4.2. División train/test


In [ ]:
X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

print(f"Train: {X_train.shape} -> {X_train.shape[0]} pingüinos")
print(f"Test: {X_test.shape} -> {X_test.shape[0]} pingüinos")


**P23.** El dataset se divide antes de escalar para evitar `data leakage`. El scaler debe aprender sus parámetros únicamente con los datos de entrenamiento. Si se ajusta usando todo el dataset, también estaría usando información del conjunto de prueba.


## Actividad 4.3. Aplicación de escaladores


In [ ]:
mm_scaler = MinMaxScaler()
zs_scaler = StandardScaler()
rb_scaler = RobustScaler()

X_train_mm = mm_scaler.fit_transform(X_train)
X_train_zs = zs_scaler.fit_transform(X_train)
X_train_rb = rb_scaler.fit_transform(X_train)

X_test_mm = mm_scaler.transform(X_test)
X_test_zs = zs_scaler.transform(X_test)
X_test_rb = rb_scaler.transform(X_test)

print("Parámetros MinMaxScaler:")
print("Mínimos:", mm_scaler.data_min_.round(2))
print("Máximos:", mm_scaler.data_max_.round(2))

print("\nParámetros StandardScaler:")
print("Medias:", zs_scaler.mean_.round(2))
print("Std:", zs_scaler.scale_.round(2))

print("\nParámetros RobustScaler:")
print("Medianas:", rb_scaler.center_.round(2))
print("IQR:", rb_scaler.scale_.round(2))


## Verificación de parámetros


In [ ]:
# Comparación Min-Max manual con los parámetros aprendidos en train
min_train_manual = X_train.min(axis=0)
max_train_manual = X_train.max(axis=0)

print("Comparación de mínimos:")
print(np.allclose(min_train_manual, mm_scaler.data_min_))

print("Comparación de máximos:")
print(np.allclose(max_train_manual, mm_scaler.data_max_))

# Verificación manual de mediana e IQR para body_mass_g
masa_train = X_train[:, 3]

mediana_manual = np.median(masa_train)
q1 = np.percentile(masa_train, 25)
q3 = np.percentile(masa_train, 75)
iqr_manual = q3 - q1

print("\nbody_mass_g")
print("Mediana manual:", mediana_manual)
print("Mediana RobustScaler:", rb_scaler.center_[3])

print("IQR manual:", iqr_manual)
print("IQR RobustScaler:", rb_scaler.scale_[3])


**P22 (segunda comparación de la guía).** Los mínimos y máximos calculados manualmente sobre `X_train` coinciden con los parámetros aprendidos por `MinMaxScaler`, porque se usa el mismo conjunto de entrenamiento.

**P23 (segunda comparación de la guía).** La mediana y el IQR calculados manualmente para `body_mass_g` coinciden con los valores aprendidos por `RobustScaler`.


# Conclusión

El preprocesamiento permite colocar variables con diferentes unidades en escalas comparables. Min-Max transforma los datos al rango de 0 a 1, Z-Score los centra respecto a la media y RobustScaler es una alternativa útil cuando existen valores extremos. También es importante dividir los datos antes de ajustar los escaladores para evitar fuga de información.
